# Feature Engineering

Feature engineering is often the difference between a mediocre model and a great one. This notebook covers:

1. **Feature Creation** - Domain-driven features, interactions, and transformations
2. **Polynomial Features** - Capturing nonlinear relationships
3. **Feature Selection** - Filter, wrapper, and embedded methods
4. **Target Encoding** - Encoding high-cardinality categoricals

**Dataset**: Ames Housing (regression) - predict house sale prices

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid")

In [ ]:
# Load Ames Housing dataset
ames = fetch_openml(name="house_prices", as_frame=True)
df = ames.frame
print(f"Shape: {df.shape}")
df.head()

## 1. Feature Creation

Good features encode domain knowledge that raw data doesn't capture directly.

In [ ]:
# Select numerical columns for demonstration
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df_num = df[num_cols].dropna()

# Domain-driven feature creation
df_feat = df_num.copy()

# Total living area (combining related features)
if "GrLivArea" in df_feat.columns and "TotalBsmtSF" in df_feat.columns:
    df_feat["TotalSF"] = df_feat["GrLivArea"] + df_feat["TotalBsmtSF"]

# Age of house at sale
if "YrSold" in df_feat.columns and "YearBuilt" in df_feat.columns:
    df_feat["HouseAge"] = df_feat["YrSold"] - df_feat["YearBuilt"]

# Total bathrooms
bath_cols = [c for c in df_feat.columns if "Bath" in c]
if bath_cols:
    df_feat["TotalBath"] = df_feat[bath_cols].sum(axis=1)

# Has feature flags (binary indicators)
if "PoolArea" in df_feat.columns:
    df_feat["HasPool"] = (df_feat["PoolArea"] > 0).astype(int)
if "GarageArea" in df_feat.columns:
    df_feat["HasGarage"] = (df_feat["GarageArea"] > 0).astype(int)

# Log transform of skewed target
df_feat["LogSalePrice"] = np.log1p(df_feat["SalePrice"])

print(f"Original features: {len(num_cols)}, After engineering: {len(df_feat.columns)}")

In [ ]:
# Compare engineered feature vs raw features
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

if "TotalSF" in df_feat.columns:
    axes[0].scatter(df_feat["GrLivArea"], df_feat["SalePrice"], alpha=0.3, s=10)
    axes[0].set_xlabel("GrLivArea (raw)")
    axes[0].set_ylabel("SalePrice")
    axes[0].set_title("Raw Feature")

    axes[1].scatter(df_feat["TotalSF"], df_feat["SalePrice"], alpha=0.3, s=10, color="teal")
    axes[1].set_xlabel("TotalSF (engineered)")
    axes[1].set_ylabel("SalePrice")
    axes[1].set_title("Engineered Feature")

plt.tight_layout()
plt.show()

## 2. Polynomial Features

Automatically create interaction and polynomial terms to capture nonlinear relationships.

In [ ]:
# Demonstrate polynomial features on a small subset
subset_cols = ["OverallQual", "GrLivArea"]
X_subset = df_feat[subset_cols].copy()

poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
X_poly = poly.fit_transform(X_subset)

poly_names = poly.get_feature_names_out(subset_cols)
print(f"Original features: {subset_cols}")
print(f"Polynomial features: {poly_names.tolist()}")
print(f"Shape: {X_subset.shape} -> {X_poly.shape}")

## 3. Feature Selection

Three main approaches:

| Method | Type | Pros | Cons |
|--------|------|------|------|
| Correlation / Mutual Info | Filter | Fast, model-agnostic | Ignores feature interactions |
| Recursive Feature Elimination | Wrapper | Considers interactions | Slow, model-dependent |
| Lasso (L1) / Feature Importance | Embedded | Built into training | Model-specific |

In [ ]:
# Prepare data
target_col = "SalePrice"
feature_cols = [c for c in df_feat.select_dtypes(include=[np.number]).columns 
                if c not in [target_col, "LogSalePrice"]]

X = df_feat[feature_cols].fillna(0)
y = df_feat[target_col]

# Method 1: Mutual Information (captures nonlinear relationships)
mi_scores = mutual_info_regression(X, y, random_state=42)
mi_df = pd.Series(mi_scores, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
mi_df.head(15).plot(kind="barh", color="teal")
plt.title("Top 15 Features by Mutual Information")
plt.xlabel("Mutual Information Score")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Method 2: Lasso (L1 regularization) - drives unimportant coefficients to zero
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

lasso = LassoCV(cv=5, random_state=42)
lasso.fit(X_train_scaled, y_train)

lasso_importance = pd.Series(np.abs(lasso.coef_), index=feature_cols).sort_values(ascending=False)
n_selected = (lasso_importance > 0).sum()
print(f"Lasso selected {n_selected}/{len(feature_cols)} features (alpha={lasso.alpha_:.2f})")

plt.figure(figsize=(10, 6))
lasso_importance.head(15).plot(kind="barh", color="coral")
plt.title("Top 15 Features by Lasso Coefficient Magnitude")
plt.xlabel("|Coefficient|")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Method 3: Tree-based feature importance
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
rf_importance.head(15).plot(kind="barh", color="steelblue")
plt.title("Top 15 Features by Random Forest Importance")
plt.xlabel("Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Domain knowledge > automated feature engineering** - understand what the data represents
2. **Log-transform skewed targets** - helps linear models and stabilizes variance
3. **Polynomial features grow fast** - degree 2 on 10 features = 65 features; use sparingly
4. **Use multiple selection methods** - features that rank high across methods are the most reliable
5. **Always engineer features before splitting** (definitions only), but fit statistics on train only